# Configuration and Project Paths

In [1]:
from pathlib import Path
import csv
import hashlib
import html
import json
import re
from collections import Counter

import pandas as pd

In [2]:
RAW_PATH = Path("../../data/raw/Software.jsonl")

PROCESSED_DIR = Path("../../data/processed")
REPORTS_DIR = Path("../../reports")

CLEAN_OUTPUT_PATH = PROCESSED_DIR / "software_clean.jsonl"
REPORT_PATH = REPORTS_DIR / "02_data_preprocessing_report.md"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
CHUNK_SIZE = 5_000
MIN_REVIEW_ROWS = 10_000
MIN_REVIEW_TEXT_LENGTH = 3

In [4]:
assert RAW_PATH.exists(), f"Input file not found: {RAW_PATH.resolve()}"

print(f"Input : {RAW_PATH.resolve()}")
print(f"Output: {CLEAN_OUTPUT_PATH.resolve()}")

Input : E:\Project\my-project\FinalYearProject\ForeSightAI\data\raw\Software.jsonl
Output: E:\Project\my-project\FinalYearProject\ForeSightAI\data\processed\software_clean.jsonl


# Detect the input format

In [5]:
SUPPORTED_EXTENSIONS = {".json", ".jsonl", ".csv", ".xls", ".xlsx"}


In [6]:
def detect_file_format(path):
    extension = path.suffix.lower()

    if extension not in SUPPORTED_EXTENSIONS:
        raise ValueError(
            f"Unsupported file format: {extension}. "
            f"Supported formats: {sorted(SUPPORTED_EXTENSIONS)}"
        )

    return extension[1:]  # Remove the leading dot from the extension

In [7]:
file_format = detect_file_format(RAW_PATH)
print("Detected format:", file_format)

Detected format: jsonl


In [8]:
TEXT_CANDIDATES = [
    "text", "reviewtext", "review_text", "review_body", "reviewbody",
    "review", "body", "content", "comment"
]

RATING_CANDIDATES = ["rating", "overall", "stars", "score"]

REVIEW_LIKE_FIELDS = {
    "text", "reviewtext", "review_text", "review_body", "reviewbody",
    "review", "body", "content", "comment", "rating", "overall",
    "stars", "score", "user_id", "reviewerid", "reviewer_id"
}

METADATA_LIKE_FIELDS = {
    "title", "brand", "price", "description", "rank", "main_cat",
    "category", "categories", "image", "images"
}

def find_column(columns, candidates):
    lookup = {str(c).strip().lower(): c for c in columns}
    for candidate in candidates:
        if candidate in lookup:
            return lookup[candidate]
    return None

# Building a common reader

In [9]:
def iter_json_records(path):
    """Yield dictionaries from JSON or JSONL input."""
    if path.suffix.lower() == ".jsonl":
        with path.open("r", encoding="utf-8") as f:
            for line_number, line in enumerate(f, start=1):
                line = line.strip()
                if not line:
                    continue
                try:
                    record = json.loads(line)
                except json.JSONDecodeError as exc:
                    raise ValueError(f"Invalid JSONL at line {line_number}: {exc}") from exc
                if not isinstance(record, dict):
                    raise ValueError(f"JSONL line {line_number} is not an object/record.")
                yield record
        return

    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, dict):
        list_values = [v for v in data.values() if isinstance(v, list)]
        if len(list_values) == 1 and all(isinstance(x, dict) for x in list_values[0]):
            yield from list_values[0]
        else:
            yield data
    elif isinstance(data, list):
        for record in data:
            if not isinstance(record, dict):
                raise ValueError("JSON array contains a non-object item.")
            yield record
    else:
        raise ValueError("JSON root must be an object or an array of objects.")

In [10]:
def iter_tabular_records(path):
    """Yield dictionaries from CSV/Excel input."""
    if path.suffix.lower() == ".csv":
        for chunk in pd.read_csv(path, chunksize=CHUNK_SIZE):
            yield from chunk.to_dict(orient="records")
    elif path.suffix.lower() in {".xls", ".xlsx"}:
        frame = pd.read_excel(path)
        yield from frame.to_dict(orient="records")
    else:
        raise ValueError(f"Unsupported tabular format: {path.suffix}")

In [11]:
def iter_records(path):
    """Unified record iterator for JSON/CSV/Excel."""
    if path.suffix.lower() in {".json", ".jsonl"}:
        yield from iter_json_records(path)
    else:
        yield from iter_tabular_records(path)


# Checking Schema Consistency

In [12]:
def is_present(value):
    if isinstance(value, (list, dict)):
        return True
    try:
        return bool(pd.notna(value))
    except (TypeError, ValueError):
        return True


In [13]:
def inspect_schema(path, max_records=None):
    key_sets = Counter()
    all_columns = set()
    rows_seen = 0

    for record in iter_records(path):
        rows_seen += 1
        all_columns.update(record.keys())
        key_set = frozenset(k for k, v in record.items() if is_present(v))
        key_sets[key_set] += 1

        if max_records is not None and rows_seen >= max_records:
            break

    return rows_seen, all_columns, key_sets



In [14]:
rows_inspected, all_columns, key_sets = inspect_schema(RAW_PATH)
text_col = find_column(all_columns, TEXT_CANDIDATES)
rating_col = find_column(all_columns, RATING_CANDIDATES)

In [15]:
print(f"Rows inspected: {rows_inspected:,}")
print(f"Distinct key-sets: {len(key_sets)}")
print(f"Text column: {text_col}")
print(f"Rating column: {rating_col}")

Rows inspected: 4,880,181
Distinct key-sets: 1
Text column: text
Rating column: rating


In [16]:
if len(key_sets) > 1:
    print("WARNING: Multiple key-sets detected.")
    print("The pipeline will preserve the union of fields and validate review-critical fields separately.")
else:
    print("Schema is structurally consistent according to the key-set check.")

Schema is structurally consistent according to the key-set check.


# Cheecking for Insufficient Data/ Metadaata

In [17]:
def validate_review_schema(
    all_columns,
    text_col,
    rating_col,
    review_like_fields,
    metadata_like_fields
):
    """
    Validate whether the dataset contains sufficient review-related fields.

    Parameters
    ----------
    all_columns : iterable
        Column names detected in the dataset.

    text_col : str or None
        Detected review-text column.

    rating_col : str or None
        Detected rating column.

    review_like_fields : set
        Known names/aliases for review-related fields.

    metadata_like_fields : set
        Known names/aliases for metadata-related fields.

    Returns
    -------
    dict
        Schema validation information.
    """

    # Normalize column names for comparison
    lower_columns = {
        str(column).lower().strip()
        for column in all_columns
    }

    review_hits = lower_columns & review_like_fields
    metadata_hits = lower_columns & metadata_like_fields

    # Basic review-data validation
    if text_col is None:
        raise ValueError(
            "No review-text column was found. "
            "The input may be metadata-only or use an unsupported schema."
        )

    if rating_col is None:
        raise ValueError(
            "No rating column was found. "
            "Confirm that the selected dataset contains review data and ratings."
        )

    return {
        "review_like_fields_found": sorted(review_hits),
        "metadata_like_fields_found": sorted(metadata_hits),
        "text_column": text_col,
        "rating_column": rating_col,
        "is_valid_review_dataset": True
    }

# Data Cleaning using BeautifulSoup

In [18]:
REMOVE_HTML_TAGS = True
VALID_RATINGS = {1, 2, 3, 4, 5}

In [19]:
try:
    from bs4 import BeautifulSoup
except ImportError:
    BeautifulSoup = None

In [20]:
def clean_html(text):
    """Remove HTML markup while retaining visible textual content."""
    text = html.unescape(str(text))
    if not REMOVE_HTML_TAGS:
        return text

    if BeautifulSoup is not None:
        return BeautifulSoup(text, "html.parser").get_text(" ")

    # Fallback for environments without BeautifulSoup.
    return re.sub(r"<[^>]*>", " ", text)

In [21]:
def normalize_whitespace(text):
    """Collapse repeated whitespace without changing meaningful wording."""
    return re.sub(r"\s+", " ", str(text)).strip()

In [22]:
def normalize_review_text(value):
    """Conservative text normalization for downstream LLM processing."""
    if value is None:
        return ""
    if pd.isna(value) if not isinstance(value, (list, dict)) else False:
        return ""

    text = str(value)
    text = clean_html(text)
    text = normalize_whitespace(text)
    return text


In [23]:
def validate_rating(value):
    """Return an integer rating in the accepted 1–5 range, otherwise None."""
    if value is None:
        return None

    if isinstance(value, bool):
        return None

    try:
        numeric = float(value)
    except (TypeError, ValueError):
        return None

    if numeric.is_integer() and int(numeric) in VALID_RATINGS:
        return int(numeric)

    return None

# Language Detection

In [24]:
LANGUAGE_FILTER_ENABLED = True
TARGET_LANGUAGE = "en"
KEEP_UNDETECTABLE_LANGUAGE = False

In [25]:
try:
    from langdetect import detect, LangDetectException
except ImportError:
    detect = None
    LangDetectException = Exception

In [26]:
def detect_language(text):
    if not text or not LANGUAGE_FILTER_ENABLED:
        return TARGET_LANGUAGE

    if detect is None:
        raise ImportError(
            "langdetect is required when LANGUAGE_FILTER_ENABLED=True. "
            "Install it or disable language filtering explicitly."
        )

    try:
        return detect(text)
    except LangDetectException:
        return None


In [27]:
def language_is_acceptable(text):
    if not LANGUAGE_FILTER_ENABLED:
        return True, TARGET_LANGUAGE

    language = detect_language(text)
    if language is None:
        return KEEP_UNDETECTABLE_LANGUAGE, None

    return language == TARGET_LANGUAGE, language

# Duplicate detection

In [28]:
print(sorted(all_columns))

['asin', 'helpful_vote', 'images', 'parent_asin', 'rating', 'text', 'timestamp', 'title', 'user_id', 'verified_purchase']


In [29]:
import pandas as pd

RAW_PATH = Path("../../data/raw/Software.jsonl")

sample_df = pd.read_json(
    RAW_PATH,
    lines=True,
    nrows=20
)

print(sample_df.columns.tolist())

['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']


In [30]:
sample_df[[
    "asin",
    "parent_asin",
    "user_id",
    "rating",
    "title",
    "text",
    "timestamp",
    "verified_purchase"
]].head(20)

,asin,parent_asin,user_id,rating,title,text,timestamp,verified_purchase
0,B07BFS3G7P,B0BQSK9QCF,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,1,malware,mcaffee IS malware,2019-07-03 19:37:12.076,False
1,B00CTQ6SIG,B00CTQ6SIG,AHSPLDNW5OOUK2PLH7GXLACFBZNQ,5,Lots of Fun,I love playing tapped out because it is fun to...,2015-02-16 20:58:56.000,True
2,B0066WJLU6,B0066WJLU6,AHSPLDNW5OOUK2PLH7GXLACFBZNQ,5,Light Up The Dark,I love this flashlight app! It really illumin...,2013-03-04 12:14:27.000,True
3,B00KCYMAWK,B00KCYMAWK,AH6CATODIVPVUOJEWHRSRCSKAOHA,4,Fun game,One of my favorite games,2019-06-20 20:10:28.662,True
4,B00P1RK566,B00P1RK566,AEINY4XOINMMJCK5GZ3M6MMHBN6A,4,I am not that good at it but my kids are,Cute game. I am not that good at it but my kid...,2014-12-11 00:19:56.000,True
5,B00CWY76CC,B00CWY76CC,AEINY4XOINMMJCK5GZ3M6MMHBN6A,4,good game,"Made me think , variety of the puzzles kept it...",2013-07-28 06:53:07.000,True
6,B018IOV40E,B018IOV40E,AEJDETWITK2KGACH7FUBMY33PPSQ,5,My favorite showVoice my favorite show,See the voice anytime my My favorite show,2018-03-07 20:56:00.247,True
7,B00EZPXYP4,B00EZPXYP4,AFSKPY37N3C43SOI5IEXEK5JSIYA,5,Great Antivirus product,Not sure what else can be said about Norton pr...,2013-11-13 15:55:13.000,False
8,B002I7PGT8,B002I7PGT8,AFSKPY37N3C43SOI5IEXEK5JSIYA,1,Fraught with too many problems,Save your money and purchase a good (i.e. Nort...,2013-04-19 13:38:15.000,False
9,B0040GFFGO,B005ZKC4FO,AFSKPY37N3C43SOI5IEXEK5JSIYA,4,Norton Internet Security,I always use Norton as my PC security applicat...,2013-04-11 09:08:23.000,False


In [31]:
sample_df[
    [
        "asin",
        "parent_asin",
        "user_id",
        "rating",
        "title",
        "text",
        "timestamp"
    ]
].isna().sum()

asin           0
parent_asin    0
user_id        0
rating         0
title          0
text           0
timestamp      0
dtype: int64

In [32]:
duplicate_columns = [
    "asin",
    "user_id",
    "rating",
    "title",
    "text"
]

print(
    "Exact duplicate rows in sample:",
    sample_df.duplicated(subset=duplicate_columns).sum()
)

Exact duplicate rows in sample: 0


In [33]:
duplicate_rows = sample_df[
    sample_df.duplicated(
        subset=duplicate_columns,
        keep=False
    )
]

duplicate_rows[
    [
        "asin",
        "user_id",
        "rating",
        "title",
        "text",
        "timestamp"
    ]
]

,asin,user_id,rating,title,text,timestamp


In [34]:
# Normalization

def normalize_for_duplicate(value):
    if pd.isna(value):
        return ""

    value = str(value)

    # Normalize whitespace
    value = re.sub(r"\s+", " ", value)

    # Remove leading/trailing whitespace
    return value.strip()

In [35]:
sample_df["title_normalized"] = (
    sample_df["title"]
    .apply(normalize_for_duplicate)
)

sample_df["text_normalized"] = (
    sample_df["text"]
    .apply(normalize_for_duplicate)
)

In [36]:
# duplicate test

duplicate_columns = [
    "asin",
    "user_id",
    "rating",
    "title_normalized",
    "text_normalized"
]

duplicate_count = sample_df.duplicated(
    subset=duplicate_columns
).sum()

print("Exact duplicates in sample:", duplicate_count)

Exact duplicates in sample: 0


In [37]:
# artificial duplicate test

test_df = sample_df.copy()

test_df = pd.concat(
    [
        test_df,
        test_df.iloc[[0]]
    ],
    ignore_index=True
)

In [38]:
duplicate_count = test_df.duplicated(
    subset=duplicate_columns
).sum()

print("Duplicates after adding test duplicate:", duplicate_count)

Duplicates after adding test duplicate: 1


## Duplicate Detection on 10000 records

In [39]:
TEST_ROWS = 10_000

test_df = pd.read_json(
    RAW_PATH,
    lines=True,
    nrows=TEST_ROWS
)

print(f"Fresh test dataset: {len(test_df):,}")

Fresh test dataset: 10,000


In [40]:
test_df["title_normalized"] = (
    test_df["title"].apply(normalize_for_duplicate)
)

test_df["text_normalized"] = (
    test_df["text"].apply(normalize_for_duplicate)
)

In [41]:
duplicate_columns = [
    "asin",
    "user_id",
    "rating",
    "title_normalized",
    "text_normalized"
]

duplicate_mask = test_df.duplicated(
    subset=duplicate_columns,
    keep="first"
)

duplicate_count = duplicate_mask.sum()

clean_test_df = test_df.loc[~duplicate_mask].copy()

print(f"Before : {len(test_df):,}")
print(f"Removed: {duplicate_count:,}")
print(f"After  : {len(clean_test_df):,}")

Before : 10,000
Removed: 2
After  : 9,998


In [42]:
duplicate_rows[
    [
        "asin",
        "user_id",
        "rating",
        "title",
        "text",
        "timestamp"
    ]
].head(20)

,asin,user_id,rating,title,text,timestamp


In [43]:
before_count = len(test_df)

duplicate_mask = test_df.duplicated(
    subset=duplicate_columns,
    keep="first"
)

duplicate_count = duplicate_mask.sum()

clean_test_df = test_df.loc[~duplicate_mask].copy()

after_count = len(clean_test_df)

print(f"Records before deduplication : {before_count:,}")
print(f"Duplicate records removed    : {duplicate_count:,}")
print(f"Records after deduplication  : {after_count:,}")

Records before deduplication : 10,000
Duplicate records removed    : 2
Records after deduplication  : 9,998


## Checking whether whitespace normalization actually matters.

In [44]:
test_copy = test_df.iloc[[0]].copy()

test_copy["text"] = (
    "   " + str(test_copy.iloc[0]["text"]) + "   "
)

test_copy["title"] = (
    "  " + str(test_copy.iloc[0]["title"]) + "  "
)

In [45]:
test_copy["title_normalized"] = (
    test_copy["title"].apply(normalize_for_duplicate)
)

test_copy["text_normalized"] = (
    test_copy["text"].apply(normalize_for_duplicate)
)

In [46]:
original = test_df.iloc[0]

print(
    original["title_normalized"]
    ==
    test_copy.iloc[0]["title_normalized"]
)

print(
    original["text_normalized"]
    ==
    test_copy.iloc[0]["text_normalized"]
)

True
True
